# 01 — Calibrating the FT232H + MPU-6050

This clean teaching notebook reconstructs your original calibration concepts:
**ellipsoid accelerometer calibration**, **six-pose body alignment**,
**stationary gyro bias**, and optional **gyro-axis alignment**.

The synthetic example runs without hardware. For real calibration, set
`RAW_STATIC_ORIENTATIONS` to a CSV with stationary samples from many
different orientations, captured using `record_ftdi.py` *without* `--calibration`.
The CSV contains SI units, so convert the accelerometer back to g before fitting.
For gyro bias, record a separate 20 s sensor-still CSV. A real six-pose
fit requires *labeled* +X/-X/+Y/-Y/+Z/-Z resting poses, not unlabeled data.
All cells catch Ctrl+C. No FTDI connection is held open here.

In [ ]:
try:
    from pathlib import Path
    import numpy as np
    import pandas as pd
    from imu_lab.calibration import (Calibration, fit_accelerometer_ellipsoid,
        estimate_six_pose_rotation, estimate_gyro_bias, fit_gyro_axis_matrix)
    from imu_lab.core import GRAVITY
    RAW_STATIC_ORIENTATIONS = None  # e.g. Path('data/many_static_orientations.csv')
    RAW_STILL_GYRO = None  # e.g. Path('data/gyro_still.csv')
    rng = np.random.default_rng(12)
    if RAW_STATIC_ORIENTATIONS is None:
        # DEMO: many known resting orientations with synthetic scale and bias.
        unit = rng.normal(size=(800, 3))
        unit /= np.linalg.norm(unit, axis=1, keepdims=True)
        acc_g = (unit / [1.02, .98, 1.03]) + [.06, -.04, .02]
        print('Using SYNTHETIC calibration readings (not from your hardware).')
    else:
        df = pd.read_csv(RAW_STATIC_ORIENTATIONS)
        acc_g = df[['ax','ay','az']].to_numpy() / GRAVITY
    print('Static orientation samples:', len(acc_g))
except KeyboardInterrupt:
    print("Cell interrupted safely; kernel stays usable.")

In [ ]:
try:
    bias, L = fit_accelerometer_ellipsoid(acc_g)
    corrected = (acc_g - bias) @ L.T
    print('Accelerometer bias [g]:', bias)
    print('Triangular calibration L:\n', L)
    print('Mean |a| before correction [g]:', np.mean(np.linalg.norm(acc_g, axis=1)))
    print('Mean |a| after correction [g]:', np.mean(np.linalg.norm(corrected, axis=1)))
except KeyboardInterrupt:
    print("Cell interrupted safely; kernel stays usable.")

## Alignment to a chosen body coordinate frame

A fitted ellipsoid becomes spherical, but this **does not identify sensor
mounting orientation**. Place the device in six *labeled* positions and
supply each stationary mean (already corrected by `L`). The vectors
below are synthetic identity-aligned placeholders. Replace the six rows
with actual means from your six labeled poses. The ordering is explicitly
`[+X,-X,+Y,-Y,+Z,-Z]` for both arrays.

In [ ]:
try:
    body_dirs = np.array([[1,0,0],[-1,0,0],[0,1,0],[0,-1,0],[0,0,1],[0,0,-1]], float)
    # DEMO ONLY. Replace with measured six means after ellipsoid correction.
    measured_pose_means = body_dirs.copy()
    R_body = estimate_six_pose_rotation(measured_pose_means, body_dirs)
    print('Body alignment:\n', R_body)
except KeyboardInterrupt:
    print("Cell interrupted safely; kernel stays usable.")

## Gyroscope bias and optional axis mapping

The sensor must remain motionless for the entire gyro-bias capture.
If no stationary file is available, the demo uses a known bias. For gyro
axis mapping, perform separate rotations about each **known positive**
body axis. The original notebook used rate-based vector alignment;
this teaches **axis alignment**, not accurate gyro scale calibration.

In [ ]:
try:
    if RAW_STILL_GYRO is None:
        still_dps = rng.normal([.12, -.05, .20], .015, size=(1000, 3))
    else:
        still_dps = np.rad2deg(pd.read_csv(RAW_STILL_GYRO)[['gx','gy','gz']].to_numpy())
    gyro_bias = estimate_gyro_bias(still_dps)
    # Replace rows with positive body X/Y/Z mean rotation rates after subtracting bias.
    gyro_axis_means = np.eye(3) * 100.0  # DEMO ONLY
    gyro_matrix = fit_gyro_axis_matrix(gyro_axis_means)
    params = Calibration(bias, L, R_body, gyro_bias, gyro_matrix).validate()
    print('Estimated gyro bias [degrees/s]:', gyro_bias)
    print('Gyro axes -> body transform:\n', gyro_matrix)
except KeyboardInterrupt:
    print("Cell interrupted safely; kernel stays usable.")

## Save a validated parameter set

This example writes to `data/calibration_demo.json`. **Do not reuse the
synthetic calibration for real recording.** Once all inputs are real,
save as `data/calibration.json` and pass to `record_ftdi.py`.
Your existing notebook's pickle keys map as follows:
`b → accel_bias_g`, `L → accel_matrix`, `aRws → accel_body_rotation`,
`w_offset → gyro_bias_dps`, `w_rotation → gyro_matrix`.
Do **not** unpickle calibration files from untrusted sources.

In [ ]:
try:
    output = Path('data/calibration_demo.json')
    output.parent.mkdir(exist_ok=True)
    params.to_json(output)
    print('Saved', output)
    print('Warning: DEMO values must not be used on a real MPU-6050.')
except KeyboardInterrupt:
    print("Cell interrupted safely; kernel stays usable.")